# TomatoCare — Phase 2: UniRepLKNet-A head training
Enable **Settings → Accelerator → GPU** (T4/P100 as available) and **Internet**. Attach dataset `thanhhainguyn/tomatocare-tomato-leaf-disease-4-classes` using **Add Input**.

This notebook expects the Phase 2 files to be published to the repository first. Alternatively attach a source-code dataset containing this checkout and set `REPO_DIR` to its location. No DINOv3 is used.

The dataset has exact-file SHA-256 deduplication only. Leaf identity (`leaf_id`) is not guaranteed independent across splits, so reported test metrics may overestimate performance on unseen leaves.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/TomatoCare")  # Or an attached source-code checkout.
if not (REPO_DIR / ".git").exists() and not (REPO_DIR / "tomatocare/train.py").exists():
    subprocess.run(["git", "clone", "https://github.com/NguyenThanhHaii/TomatoCare.git", str(REPO_DIR)], check=True)
if not (REPO_DIR / "tomatocare/train.py").is_file():
    raise RuntimeError("Phase 2 code is missing. Publish the reviewed changes or attach the complete source checkout.")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))
import torch
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable the Kaggle GPU accelerator before training"
print(torch.cuda.get_device_name(0))


In [ ]:
from tomatocare.data import discover_bundle, LeafDataset
from torch.utils.data import DataLoader

DATA_ROOT = "/kaggle/input"
# Set these only if metadata has ambiguous or nonstandard column names.
PATH_COLUMN = None
LABEL_COLUMN = None
SPLIT_COLUMN = None
bundle = discover_bundle(DATA_ROOT, PATH_COLUMN, LABEL_COLUMN, SPLIT_COLUMN)
print("Bundle:", bundle.root)
print("Logit order from labels.json:", dict(enumerate(bundle.class_names)))
counts = {split: len(rows) for split, rows in bundle.records.items()}
print("Split counts:", counts)
assert counts == {"train": 3816, "val": 816, "test": 820}
images, targets = next(iter(DataLoader(LeafDataset(bundle.records["train"], train=True), batch_size=4)))
print("Training batch:", images.shape, targets.tolist())
assert images.shape == (4, 3, 224, 224)


## Train five epochs
The default downloads the official ImageNet-1K checkpoint. If Internet is disabled or download fails, attach the official `unireplknet_a_in1k_224_acc77.03.pth` as another Kaggle input and set `PRETRAINED_CHECKPOINT` below to its actual path. Loading failures stop the run; there is no random-initialization fallback.

Only the classification head is optimized. Backbone parameters and BatchNorm statistics stay frozen. Start with batch size 32; use 16 if the GPU reports out-of-memory. Choose settings before running the final test. Validation selects the best Macro-F1 checkpoint; the script evaluates test once afterward. Do not repeatedly rerun with test-driven settings. Existing experiment outputs must be archived before a deliberate new run.


In [ ]:
from tomatocare.train import TrainConfig, run_training

PRETRAINED_CHECKPOINT = None  # Or the actual attached official checkpoint path.
config = TrainConfig(
    data_root=DATA_ROOT, output_dir="/kaggle/working",
    pretrained_checkpoint=PRETRAINED_CHECKPOINT,
    epochs=5, batch_size=32, num_workers=2, device="cuda",
    path_column=PATH_COLUMN, label_column=LABEL_COLUMN, split_column=SPLIT_COLUMN,
)
checkpoint_path, test_metrics = run_training(config)
print("Checkpoint:", checkpoint_path)


In [ ]:
import json
from IPython.display import Image, display

# Display saved results; this does not run another test evaluation.
results = json.loads(Path("/kaggle/working/test_metrics.json").read_text())
for key in ("accuracy", "precision_macro", "recall_macro", "macro_f1"):
    print(f"{key}: {results[key]:.4f}")
print(json.dumps(results["classification_report"], indent=2))
display(Image(filename="/kaggle/working/test_confusion_matrix.png"))


## Save outputs
Use **Save Version → Save & Run All** for one clean final execution, rather than rerunning the held-out test in a live session. After completion, download `best_unireplknet_a.pth`, `history.json`, `test_metrics.json`, `validation_metrics.json`, and the confusion-matrix CSV/PNG files from the notebook **Output** tab. The checkpoint includes `state_dict`, logit-order label mapping, preprocessing, and best validation epoch. Never commit checkpoint files to GitHub.

No dataset training metrics are supplied in this notebook before execution.
